In [1]:
import pandas as pd

df= pd.read_csv('/content/dublin_bikes_all.csv')
df.shape
df.head(5)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 9 columns):
 #   Column               Non-Null Count    Dtype  
---  ------               --------------    -----  
 0   last_reported        1048575 non-null  object 
 1   station_id           1048575 non-null  int64  
 2   num_bikes_available  1048575 non-null  int64  
 3   num_docks_available  1048575 non-null  int64  
 4   name                 1048575 non-null  object 
 5   address              1048575 non-null  object 
 6   lat                  1048575 non-null  float64
 7   lon                  1048575 non-null  float64
 8   capacity             1048575 non-null  int64  
dtypes: float64(2), int64(4), object(3)
memory usage: 72.0+ MB


In [2]:
df.duplicated().sum()

np.int64(0)

In [6]:
df.isna().sum().sort_values(ascending=False).head()

,0
last_reported,0
station_id,0
num_bikes_available,0
num_docks_available,0
name,0


No missing value

In [5]:
num = df.select_dtypes(include="number")
cat= df.select_dtypes(exclude = "number")

Split the dataset into numerical and categorical data

In [9]:
num.describe().T

,count,mean,std,min,25%,50%,75%,max
station_id,1048575.0,60.365841,36.445486,1.00000,30.00000,60.00000,90.00000,507.00000
num_bikes_available,1048575.0,12.501497,9.069503,0.00000,5.00000,11.00000,18.00000,40.00000
num_docks_available,1048575.0,19.326606,10.714019,0.00000,11.00000,19.00000,28.00000,40.00000
lat,1048575.0,53.345631,0.007644,53.33010,53.33980,53.34520,53.35100,53.36000
lon,1048575.0,-6.264334,0.018072,-6.31002,-6.27523,-6.26298,-6.25156,-6.23085
capacity,1048575.0,32.005430,7.611704,1.00000,29.00000,30.00000,40.00000,40.00000


Converting last reported to a datetime feature

In [12]:
df["last_reported"] = pd.to_datetime(df["last_reported"], errors="coerce", utc=True)
df[["last_reported"]].isna().mean()


,0
last_reported,0.573158


Checking for negativity

In [15]:

(df[["num_bikes_available","num_docks_available","capacity"]] < 0).sum()


,0
num_bikes_available,0
num_docks_available,0
capacity,0


number of bikes available + docks available should be equal to the capacity

In [16]:
(df["num_bikes_available"] + df["num_docks_available"] - df["capacity"]).describe()

,0
count,1.048575e+06
mean,-1.773273e-01
std,1.922420e+00
min,-4.000000e+01
25%,0.000000e+00
50%,0.000000e+00
75%,0.000000e+00
max,0.000000e+00


min =-40 , so in worst case the bikes + docks is 40 less than capacity. Meaning some docks are missing

In [17]:
diff = df["num_bikes_available"] + df["num_docks_available"] - df["capacity"]
diff.value_counts().head(10)

df.assign(diff=diff).query("diff < 0").groupby("station_id")["diff"].agg(["count","min","mean"]).sort_values("min").head(10)


,count,min,mean
station_id,,,
53,378,-40,-9.962963
114,556,-40,-7.928058
113,502,-40,-14.754980
111,297,-40,-5.239057
102,168,-40,-20.333333
89,294,-40,-13.017007
75,627,-40,-4.526316
71,272,-40,-13.143382
87,1120,-38,-11.451786


These stations have docks which are unusable or unavailable

In [18]:
station_static = ["name","address","lat","lon","capacity"]
df.groupby("station_id")[station_static].nunique().sort_values(by="name", ascending=False).head()


,name,address,lat,lon,capacity
station_id,,,,,
1,1,1,1,1,1
2,1,1,1,1,1
3,1,1,1,1,1
4,1,1,1,1,1
5,1,1,1,1,1


Checked if there is more than 1 unique value for name, address, latitude, longitude and capacity

##Checking the regularity of collected data

In [20]:
df = df.sort_values(["station_id","last_reported"])
delta = df.groupby("station_id")["last_reported"].diff().dt.total_seconds()/60
delta.value_counts().head(10),


(last_reported
 30.0       446219
 34590.0       660
 33150.0       440
 30270.0       110
 40350.0        12
 38910.0         9
 43230.0         6
 41790.0         4
 36030.0         2
 3630.0          1
 Name: count, dtype: int64,)

Most of the stations(446219) have 30 min of regularity. But some have high irregularity

In [21]:
delta.describe()

,last_reported
count,447463.000000
mean,123.961735
std,1782.348403
min,30.000000
25%,30.000000
50%,30.000000
75%,30.000000
max,43230.000000


Normalizing the bike availability

In [22]:
df["bikes_norm"] = df["num_bikes_available"] / df["capacity"]


In [23]:
df["empty"] = (df["num_bikes_available"] == 0)
df["full"]  = (df["num_docks_available"] == 0)


In [24]:
df["hour"] = df["last_reported"].dt.hour
df["dow"]  = df["last_reported"].dt.dayofweek


Grouped the data by hour and day of week

In [27]:
empty_by_hour = df.groupby("hour")["empty"].mean()
empty_by_dow  = df.groupby("dow")["empty"].mean()
print(empty_by_hour)
print(empty_by_dow)

hour
0.0     0.072447
1.0     0.071000
2.0     0.070249
3.0     0.070034
4.0     0.070034
5.0     0.066924
6.0     0.050622
7.0     0.046654
8.0     0.050837
9.0     0.066441
10.0    0.071858
11.0    0.070996
12.0    0.066059
13.0    0.060054
14.0    0.047936
15.0    0.047292
16.0    0.047668
17.0    0.051582
18.0    0.056032
19.0    0.057962
20.0    0.061930
21.0    0.066059
22.0    0.069169
23.0    0.071210
Name: empty, dtype: float64
dow
0.0    0.073284
1.0    0.053123
2.0    0.063289
3.0    0.064284
4.0    0.060667
5.0    0.060998
6.0    0.056327
Name: empty, dtype: float64


This shows what percentage of the docks are empty at every hour (across all stations) and at every day

##Example

In [28]:
print(empty_by_hour[8])

0.05083655083655084


This means 5 % of all docks were empty at 8 am

In [29]:
full_by_hour = df.groupby("hour")["full"].mean()
full_by_dow  = df.groupby("dow")["full"].mean()
print(full_by_hour)
print(full_by_dow)

hour
0.0     0.043919
1.0     0.050032
2.0     0.050890
3.0     0.050997
4.0     0.051051
5.0     0.045367
6.0     0.029387
7.0     0.020163
8.0     0.018876
9.0     0.028582
10.0    0.033462
11.0    0.030726
12.0    0.026971
13.0    0.024343
14.0    0.024504
15.0    0.024182
16.0    0.025469
17.0    0.028954
18.0    0.034638
19.0    0.035121
20.0    0.040429
21.0    0.042895
22.0    0.043110
23.0    0.041450
Name: full, dtype: float64
dow
0.0    0.033430
1.0    0.034274
2.0    0.035490
3.0    0.035030
4.0    0.044674
5.0    0.031438
6.0    0.032290
Name: full, dtype: float64


Ranking the worst stations:

In [32]:
station_empty = df.groupby("station_id")["empty"].mean().sort_values(ascending=False).head(15)
station_full  = df.groupby("station_id")["full"].mean().sort_values(ascending=False).head(15)
print(station_empty)


station_id
20     0.361477
1      0.262587
27     0.183943
61     0.180578
99     0.171692
114    0.169258
103    0.161959
47     0.156141
15     0.152316
30     0.150095
62     0.147149
25     0.136133
89     0.132127
65     0.126415
41     0.124603
Name: empty, dtype: float64


In [33]:
print(station_full)

station_id
507    1.000000
1      0.232636
10     0.204041
115    0.183751
100    0.176135
42     0.171250
43     0.112016
20     0.111025
33     0.102285
34     0.100592
91     0.098805
40     0.098371
9      0.090332
60     0.089919
31     0.088217
Name: full, dtype: float64
